# Query Projector Training

This notebook trains a small MLP that projects e5 query embeddings (1024D) into the GNN embedding space (256D),
so that queries can be directly compared to function GNN embeddings via cosine similarity.

**Self-supervised training**: each function's `combinedName + docstring` text is treated as the query side,
and its GNN embedding is the target. Trained with InfoNCE contrastive loss.

In [1]:
import os
import sys
import torch

sys.path.append(os.path.abspath('.'))

from src.utils.query_projector import (
    QueryProjector,
    collect_training_data,
    train_projector,
    load_projector,
)

c:\Users\janos\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

In [2]:
GNN_EMB_PATH = '../code-knowledge-graph/gnn_out/gnn_embeddings.pkl'
OUT_DIR = 'projector_out'
ENCODER = 'intfloat/e5-large-v2'

NEO4J_URI = 'bolt://127.0.0.1:7687'
NEO4J_USER = 'neo4j'
NEO4J_PASSWORD = 'password'

EPOCHS = 100
BATCH_SIZE = 256
LR = 1e-3
TEMPERATURE = 0.05

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Device: {DEVICE}')

Device: cuda


## 2. Collect Training Data

Fetches function texts from Neo4j, loads GNN embeddings from pickle,
and encodes the texts with e5-large-v2. Slowest cell — ~2-5 min on CPU, faster on GPU.

In [3]:
X, Y, fids = collect_training_data(
    neo4j_uri=NEO4J_URI,
    user=NEO4J_USER,
    password=NEO4J_PASSWORD,
    gnn_emb_path=GNN_EMB_PATH,
    encoder_model=ENCODER,
    device=DEVICE,
)
print(f'Training pairs: {X.size(0):,}')
print(f'Query embedding dim:  {X.size(-1)}')
print(f'Target embedding dim: {Y.size(-1)}')


Training pairs: 10,484
Query embedding dim:  1024
Target embedding dim: 256


## 3. Training

Val top-1 accuracy: percentage of validation queries where the projected vector is closest
to its own function's GNN embedding. Random baseline is `1/N`, perfect score is 1.0.

In [4]:
projector, best_acc, history = train_projector(
    X=X,
    Y=Y,
    out_dir=OUT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    temperature=TEMPERATURE,
    device=DEVICE,
)
print(f'\nBest val top-1 accuracy: {best_acc:.4f}')
print(f'Checkpoint saved to: {os.path.join(OUT_DIR, "projector.pt")}')

Epoch 001 | loss=5.1798 | val_top1=0.0076 *
Epoch 005 | loss=4.4231 | val_top1=0.0181 *
Epoch 010 | loss=4.2780 | val_top1=0.0286
Epoch 015 | loss=4.1913 | val_top1=0.0334
Epoch 020 | loss=4.1871 | val_top1=0.0344
Epoch 025 | loss=4.1453 | val_top1=0.0372 *
Epoch 030 | loss=4.1148 | val_top1=0.0267
Epoch 035 | loss=4.0822 | val_top1=0.0286
Epoch 040 | loss=4.0646 | val_top1=0.0324
Epoch 045 | loss=4.0506 | val_top1=0.0277
Epoch 050 | loss=4.0325 | val_top1=0.0219
Epoch 055 | loss=4.0130 | val_top1=0.0344
Epoch 060 | loss=3.9934 | val_top1=0.0286
Epoch 065 | loss=3.9938 | val_top1=0.0286
Epoch 070 | loss=3.9993 | val_top1=0.0305
Epoch 075 | loss=3.9815 | val_top1=0.0248
Epoch 080 | loss=3.9981 | val_top1=0.0305
Epoch 085 | loss=3.9827 | val_top1=0.0239
Epoch 090 | loss=3.9734 | val_top1=0.0267
Epoch 095 | loss=3.9590 | val_top1=0.0229
Epoch 100 | loss=3.9609 | val_top1=0.0296

Best val top-1 accuracy: 0.0372
Checkpoint saved to: projector_out\projector.pt


In [ ]:
import json

history_path = os.path.join(OUT_DIR, 'projector_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f)
print(f'History saved to: {history_path}')

## 4. Smoke Test

Load the projector, encode a sample query, and find the top-5 closest functions in GNN space.

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from src.utils.query_projector import project_query

encoder = HuggingFaceEmbeddings(model_name=ENCODER, model_kwargs={'device': DEVICE})
p = load_projector(os.path.join(OUT_DIR, 'projector.pt'), device=DEVICE)

QUERY = 'How does StandardScaler handle missing values in input data?'
z_query = project_query(p, QUERY, encoder, device=DEVICE)

Y_dev = Y.to(DEVICE)
sims = (Y_dev @ z_query)
top5 = sims.topk(5)

print(f'Query: {QUERY}\n')
print('Top 5 closest functions (by GNN-space cosine similarity):')
for rank, (sim, idx) in enumerate(zip(top5.values.tolist(), top5.indices.tolist()), 1):
    print(f'  {rank}. fid={fids[idx]:>6} | sim={sim:.4f}')

Query: How does StandardScaler handle missing values in input data?

Top 5 closest functions (by GNN-space cosine similarity):
  1. fid=   512 | sim=0.3930
  2. fid=   225 | sim=0.3919
  3. fid=   203 | sim=0.3905
  4. fid=  1583 | sim=0.3897
  5. fid=   191 | sim=0.3889
